In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.filters import threshold_otsu

# --- CONFIG ---
OUTPUT_DIR = "/Users/aditiiyer/Documents/Thesis/Extent_EDA"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Ensure correct date type
df_sg = pd.read_csv("/Users/aditiiyer/Documents/Thesis/SG_cols.csv")
df["date"] = pd.to_datetime(df["date"]).dt.to_period("M").dt.to_timestamp()

# --- STORAGE ---
monthly_results = []

# --- PROCESS MONTHLY ---
for month, g in df.groupby("date"):
    # Drop NaNs for threshold calculation
    ndvi_vals = g["NDVI_sg"].dropna().values
    ndwi_vals = g["NDWI_sg"].dropna().values
    
    if len(ndvi_vals) == 0 or len(ndwi_vals) == 0:
        continue  # skip months with no data
    
    # --- Otsu thresholds ---
    ndvi_thresh = threshold_otsu(ndvi_vals)
    ndwi_thresh = threshold_otsu(ndwi_vals)
    
    # --- Classification ---
    wetland_flag = g["NDVI_sg"] > ndvi_thresh
    lake_flag    = g["NDWI_sg"] > ndwi_thresh
    
    wetland_frac = wetland_flag.mean()
    lake_frac    = lake_flag.mean()
    
    monthly_results.append({
        "date": month,
        "ndvi_otsu_thresh": ndvi_thresh,
        "ndwi_otsu_thresh": ndwi_thresh,
        "wetland_frac": wetland_frac,
        "lake_frac": lake_frac
    })

# --- TO DATAFRAME ---
extent_monthly = pd.DataFrame(monthly_results).sort_values("date")

# --- SAVE CSV ---
extent_monthly.to_csv(os.path.join(OUTPUT_DIR, "wetland_lake_extent_otsu.csv"), index=False)

# --- PLOT ---
plt.figure(figsize=(10, 4.5), dpi=300)
plt.plot(extent_monthly["date"], extent_monthly["wetland_frac"], label="Wetland extent (NDVI Otsu)", linewidth=1.8)
plt.plot(extent_monthly["date"], extent_monthly["lake_frac"], label="Lake extent (NDWI Otsu)", linewidth=1.8)
plt.xlabel("Month")
plt.ylabel("Fraction of pixels")
plt.title("Monthly Wetland & Lake Extent — Otsu Threshold (SG-smoothed)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "wetland_lake_extent_timeseries_otsu.png"), dpi=300, bbox_inches="tight")
plt.close()

In [ ]:
# ============================================
# Build multi-year wetland extent from 3 rasters
# - Inputs: annual Otsu masks (0/1), same grid/CRS
# - Outputs: union, intersection, majority(>=2/3), fractional persistence
# - Optional: polygonize majority to GeoPackage & report area
# ============================================

import os
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.features import shapes
from shapely.geometry import shape as shapely_from_geojson
import geopandas as gpd

# ---------- CONFIG ----------
IN_2022 = "/Users/aditiiyer/Documents/Thesis/Otsu_extents_yoy/2022_Otsu_extent.tif"
IN_2023 = "/Users/aditiiyer/Documents/Thesis/Otsu_extents_yoy/2023_Otsu_extent.tif"
IN_2024 = "/Users/aditiiyer/Documents/Thesis/Otsu_extents_yoy/2024_Otsu_extent.tif"

OUT_DIR = "wetland_extent_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_UNION      = os.path.join(OUT_DIR, "wetland_union.tif")
OUT_INTERSECT  = os.path.join(OUT_DIR, "wetland_intersection.tif")
OUT_MAJORITY   = os.path.join(OUT_DIR, "wetland_majority_2of3.tif")
OUT_FRACTION   = os.path.join(OUT_DIR, "wetland_fraction_0to1.tif")
OUT_GPKG       = os.path.join(OUT_DIR, "wetland_majority_2of3.gpkg")  # polygonized

# If your input rasters lack nodata in metadata, set it here. Otherwise leave as None.
FORCE_NODATA = None  # e.g., 255

# Polygonize majority?
DO_POLYGONIZE = True
GPKG_LAYER    = "wetland_majority"

# ---------- HELPERS ----------
def open_rasters(paths):
    srcs = [rasterio.open(p) for p in paths]
    # sanity checks
    ref = srcs[0]
    for s in srcs[1:]:
        assert s.width == ref.width and s.height == ref.height, "Different raster sizes."
        assert s.crs == ref.crs, "Different CRS."
        assert s.transform == ref.transform, "Different geotransforms."
    return srcs

def write_like(ref, out_path, dtype, nodata, compress="LZW"):
    profile = ref.profile.copy()
    profile.update(
        dtype=dtype,
        count=1,
        nodata=nodata,
        compress=compress,
        bigtiff="IF_SAFER"
    )
    return rasterio.open(out_path, "w", **profile)

def window_iter(width, height, block=1024):
    for row in range(0, height, block):
        h = min(block, height - row)
        for col in range(0, width, block):
            w = min(block, width - col)
            yield Window(col_off=col, row_off=row, width=w, height=h)

# ---------- MAIN ----------
paths = [IN_2022, IN_2023, IN_2024]
srcs = open_rasters(paths)  # returns list of rasterio.DatasetReader objects
ref = srcs[0]

nodatas = [FORCE_NODATA if FORCE_NODATA is not None else s.nodata for s in srcs]
if any(nd is None for nd in nodatas):
    print(" Some inputs have nodata=None; treating all values as valid (0/1). Set FORCE_NODATA if needed.")
nd = nodatas[0] if all(n == nodatas[0] for n in nodatas) else None

# Prepare writers
out_union     = write_like(ref, OUT_UNION,     dtype="uint8",  nodata=0)
out_intersect = write_like(ref, OUT_INTERSECT, dtype="uint8",  nodata=0)
out_majority  = write_like(ref, OUT_MAJORITY,  dtype="uint8",  nodata=0)
out_fraction  = write_like(ref, OUT_FRACTION,  dtype="float32", nodata=np.nan)

try:
    for win in window_iter(ref.width, ref.height, block=1024):
        arrays = []
        valid_masks = []
        for s, s_nd in zip(srcs, nodatas):
            a = s.read(1, window=win).astype(np.float32)
            arrays.append(a)
            if s_nd is None:
                vm = np.ones_like(a, dtype=bool)
            else:
                vm = ~(a == s_nd)
            valid_masks.append(vm)

        valid_all = valid_masks[0] & valid_masks[1] & valid_masks[2]

        a0 = np.where(valid_masks[0], arrays[0], 0.0)
        a1 = np.where(valid_masks[1], arrays[1], 0.0)
        a2 = np.where(valid_masks[2], arrays[2], 0.0)

        a0 = (a0 > 0.5).astype(np.uint8)
        a1 = (a1 > 0.5).astype(np.uint8)
        a2 = (a2 > 0.5).astype(np.uint8)

        ssum = a0 + a1 + a2

        union     = np.where(valid_all, (ssum >= 1).astype(np.uint8), 0)
        intersect = np.where(valid_all, (ssum == 3).astype(np.uint8), 0)
        majority  = np.where(valid_all, (ssum >= 2).astype(np.uint8), 0)
        fraction  = np.where(valid_all, (ssum / 3.0).astype(np.float32), np.nan)

        out_union.write(union, 1, window=win)
        out_intersect.write(intersect, 1, window=win)
        out_majority.write(majority, 1, window=win)
        out_fraction.write(fraction, 1, window=win)
finally:
    out_union.close()
    out_intersect.close()
    out_majority.close()
    out_fraction.close()
    for s in srcs:
        s.close()

print(f" Wrote:\n  {OUT_UNION}\n  {OUT_INTERSECT}\n  {OUT_MAJORITY}\n  {OUT_FRACTION}")

# ---------- (Optional) Polygonize majority to GeoPackage ----------
if DO_POLYGONIZE:
    import rasterio
    from rasterio.features import shapes as rio_shapes

    with rasterio.open(OUT_MAJORITY) as src:
        transform = src.transform
        crs = src.crs
        geoms = []
        attrs = []

        # iterate in one go (safe if raster isn't massive; otherwise tile here too)
        arr = src.read(1)
        for geom, val in rio_shapes(arr.astype(np.uint8), transform=transform, connectivity=4):
            if int(val) == 1:
                geoms.append(shapely_from_geojson(geom))
                attrs.append({"class":"wetland_consensus_2of3"})

    gdf = gpd.GeoDataFrame(attrs, geometry=geoms, crs=crs)
    # Optional: dissolve to a single multipart polygon
    gdf = gdf.dissolve(by="class", as_index=False)

    if os.path.exists(OUT_GPKG):
        os.remove(OUT_GPKG)
    gdf.to_file(OUT_GPKG, layer="wetland_consensus", driver="GPKG")
    print(f"Polygonized majority → {OUT_GPKG}")


In [ ]:
import geopandas as gpd
from shapely.ops import unary_union

def find_buffer_for_two_polygons(vector_path, start=0.0001, step=0.0001, max_buffer=0.01):
    """
    Iteratively buffers the geometry until exactly two polygons remain.
    
    Parameters
    ----------
    vector_path : str
        Path to input vector file (e.g., shapefile or geojson)
    start : float
        Initial buffer distance in coordinate units
    step : float
        Step size for increasing buffer
    max_buffer : float
        Maximum buffer distance to try
    
    Returns
    -------
    (float, GeoDataFrame)
        Tuple of the buffer distance and resulting polygons
    """
    gdf = gpd.read_file(vector_path)
    geom = unary_union(gdf.geometry)
    
    buffer_val = start
    result_gdf = None

    while buffer_val <= max_buffer:
        buffered = geom.buffer(buffer_val)
        buffered_gdf = gpd.GeoDataFrame(geometry=[buffered], crs=gdf.crs).explode(index_parts=False)
        n_polygons = len(buffered_gdf)
        
        if n_polygons == 2:
            result_gdf = buffered_gdf
            print(f"Found 2 polygons at buffer = {buffer_val}")
            break
        
        buffer_val += step

    if result_gdf is None:
        print("Did not reach exactly 2 polygons within max_buffer.")
    return buffer_val, result_gdf

# Example usage
vector_path = "/Users/aditiiyer/Documents/Thesis/wetland_extent_outputs/Vectorised_Intersection/Vectorised.shp"
buffer_val, two_polygons = find_buffer_for_two_polygons(vector_path)

if two_polygons is not None:
    two_polygons.to_file("wetland_two_polygons.gpkg", driver="GPKG")